# Jawaban Tugas Praktikum Association Rule Mining
Tugas 1–3 menggunakan 60 transaksi asli modul (harus disediakan); Tugas 4 menggunakan 48 transaksi sintetis kedai kopi yang didefinisikan di notebook ini. Jalankan berurutan di Google Colab.

In [ ]:
%pip -q install mlxtend pandas numpy matplotlib

## Persiapan data asli untuk Tugas 1–3
Modul hanya memuat lima transaksi pertama, bukan keseluruhan 60 transaksi. Isi `transactions_modul` dengan seluruh 60 transaksi asli sebelum menjalankan bagian ini. Jangan mengganti dengan data buatan karena hasil tidak akan cocok.

In [ ]:
transactions_modul = [
 ['Gula','Kopi','Susu'], ['Gula','Kopi','Susu'],
 ['Mentega','Roti','Susu','Telur'], ['Mie Instan'],
 ['Keju','Roti','Susu','Telur']
]
# Tambahkan transaksi T6 sampai T60 di atas sebelum menjalankan Tugas 1–3.
import pandas as pd, numpy as np, matplotlib.pyplot as plt, time
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, fpgrowth, association_rules
assert len(transactions_modul)==60, 'Diperlukan seluruh 60 transaksi asli modul, bukan hanya lima contoh.'
te=TransactionEncoder(); df=pd.DataFrame(te.fit(transactions_modul).transform(transactions_modul),columns=te.columns_)

## Tugas 1 — Eksplorasi ambang batas
Modul melaporkan pada min_support 0.10 terdapat **19 itemset dan 20 aturan**, sedangkan pada 0.15 terdapat **17 itemset dan 18 aturan** (confidence 0.5). Ambang lebih rendah menangkap lebih banyak pola, tetapi dapat meningkatkan noise. Grafik berikut dan daftar aturan gugur dihitung setelah data lengkap tersedia.

In [ ]:
for s in [0.10,0.15]:
 fi=apriori(df,min_support=s,use_colnames=True)
 rr=association_rules(fi,metric='confidence',min_threshold=0.5)
 print(f'min_support={s:.2f}: {len(fi)} itemset, {len(rr)} aturan')
fi=apriori(df,min_support=0.15,use_colnames=True)
r05=association_rules(fi,metric='confidence',min_threshold=0.5)
r07=association_rules(fi,metric='confidence',min_threshold=0.7)
fmt=lambda z: ', '.join(sorted(z))
print('Aturan bertahan:',len(r07),'dari',len(r05))
print('Aturan gugur:')
for _,r in r05[r05.confidence<0.7].iterrows(): print(fmt(r.antecedents),'->',fmt(r.consequents), 'confidence',round(r.confidence,3))
thresholds=[.5,.6,.7,.8,.9,1.]
counts=[len(association_rules(fi,metric='confidence',min_threshold=c)) for c in thresholds]
plt.plot(thresholds,counts,marker='o');plt.xticks(thresholds);plt.xlabel('min_confidence');plt.ylabel('Jumlah aturan');plt.title('Pengaruh min_confidence (min_support=0.15)');plt.grid(alpha=.3);plt.show()
print(list(zip(thresholds,counts)))

## Tugas 2 — Analisis dua aturan lain
Pilih aturan **Mie Instan → Telur** dan **Roti → Susu**, bila keduanya lolos confidence 0.5. Rumus: leverage = support(X∪Y) − support(X)×support(Y); conviction = (1−support(Y))/(1−confidence), dengan nilai tak hingga saat confidence=1. Lift >1 menunjukkan asosiasi positif, tetapi tidak membuktikan sebab-akibat.

In [ ]:
for ant,cons in [('Mie Instan','Telur'),('Roti','Susu')]:
 sub=r05[(r05.antecedents==frozenset([ant]))&(r05.consequents==frozenset([cons]))]
 if len(sub):
  r=sub.iloc[0];print(f'{ant} -> {cons}: support={r.support:.3f}, confidence={r.confidence:.3f}, lift={r.lift:.3f}, leverage={r.leverage:.4f}, conviction={r.conviction:.3f}')
 else: print(f'{ant} -> {cons}: tidak memenuhi ambang')

Interpretasi bisnis: (1) Bila Mie Instan → Telur memiliki lift >1, letakkan telur dekat mi instan dan uji paket hemat mi+telur. (2) Bila Roti → Susu memiliki lift >1, tawarkan rekomendasi susu ketika pelanggan membeli roti, lalu evaluasi dengan A/B test.

## Tugas 3 — Apriori vs FP-Growth
Pengukuran ulang pada 0.05 dan 0.15 dilakukan pada mesin yang sama. Angka modul pada 0.15 (10.42 ms vs 2.95 ms) merupakan contoh hasil modul, bukan hasil pengukuran notebook ini. Performa bisa berubah antarmesin dan antareksekusi.

In [ ]:
for s in [.05,.15]:
 measurements={}
 for name,func in [('Apriori',apriori),('FP-Growth',fpgrowth)]:
  t=time.perf_counter(); result=func(df,min_support=s,use_colnames=True); elapsed=(time.perf_counter()-t)*1000
  measurements[name]=(result,elapsed)
  print(f'support={s:.2f}, {name}: {len(result)} itemset, {elapsed:.3f} ms')
 a=set(map(frozenset,measurements['Apriori'][0].itemsets)); f=set(map(frozenset,measurements['FP-Growth'][0].itemsets))
 print('Itemset identik:',a==f,'rasio waktu Apriori/FP:',round(measurements['Apriori'][1]/measurements['FP-Growth'][1],2))

## Tugas 4 — Studi kasus mandiri: kedai kopi
Dataset **sintetis** 48 transaksi dan 10 jenis item; dibuat khusus sebagai simulasi, bukan catatan penjualan nyata. Pola kombinasi sengaja dibangun agar analisis asosiasi dapat diuji. min_support=0.15 berarti minimal 8 dari 48 transaksi; min_confidence=0.55 digunakan agar aturan tidak terlalu longgar.

In [ ]:
# Setiap grup terdiri atas 8 transaksi; 48 total, 10 item berbeda.
groups=[
 [['Kopi Susu','Croissant','Air Mineral'],['Kopi Susu','Croissant'],['Kopi Susu','Croissant','Brownies'],['Kopi Susu','Croissant','Air Mineral'],['Kopi Susu','Croissant'],['Kopi Susu','Croissant','Brownies'],['Kopi Susu','Croissant'],['Kopi Susu','Croissant','Air Mineral']],
 [['Americano','Cookies'],['Americano','Cookies','Air Mineral'],['Americano','Cookies'],['Americano','Cookies','Sandwich'],['Americano','Cookies'],['Americano','Cookies','Air Mineral'],['Americano','Cookies'],['Americano','Cookies','Sandwich']],
 [['Matcha','Cheesecake'],['Matcha','Cheesecake'],['Matcha','Cheesecake','Air Mineral'],['Matcha','Cheesecake'],['Matcha','Cheesecake'],['Matcha','Cheesecake','Brownies'],['Matcha','Cheesecake'],['Matcha','Cheesecake']],
 [['Teh Lemon','Sandwich'],['Teh Lemon','Sandwich'],['Teh Lemon','Sandwich','Air Mineral'],['Teh Lemon','Sandwich'],['Teh Lemon','Sandwich'],['Teh Lemon','Sandwich','Cookies'],['Teh Lemon','Sandwich'],['Teh Lemon','Sandwich']],
 [['Kopi Susu','Brownies'],['Kopi Susu','Brownies'],['Kopi Susu','Brownies','Air Mineral'],['Kopi Susu','Brownies'],['Kopi Susu','Brownies'],['Kopi Susu','Brownies'],['Kopi Susu','Brownies','Cookies'],['Kopi Susu','Brownies']],
 [['Americano','Sandwich'],['Americano','Sandwich'],['Americano','Sandwich','Air Mineral'],['Americano','Sandwich'],['Americano','Sandwich'],['Americano','Sandwich','Cheesecake'],['Americano','Sandwich'],['Americano','Sandwich']]
]
transactions_kopi=[t for group in groups for t in group]
items=sorted(set().union(*map(set,transactions_kopi)))
print('Transaksi:',len(transactions_kopi),'Jenis item:',len(items),items)
te4=TransactionEncoder(); df4=pd.DataFrame(te4.fit(transactions_kopi).transform(transactions_kopi),columns=te4.columns_)
fi4=apriori(df4,min_support=.15,use_colnames=True)
r4=association_rules(fi4,metric='confidence',min_threshold=.55).sort_values('lift',ascending=False)
print('Frequent itemset:',len(fi4),'Aturan:',len(r4))
show=r4[['antecedents','consequents','support','confidence','lift','leverage','conviction']].copy()
for c in ['antecedents','consequents']:show[c]=show[c].map(lambda s:', '.join(sorted(s)))
print(show.head(15).round(3).to_string(index=False))

In [ ]:
plt.scatter(r4.support,r4.confidence,s=90*r4.lift,c=r4.lift,alpha=.75)
plt.colorbar(label='Lift');plt.xlabel('Support');plt.ylabel('Confidence');plt.title('Kedai Kopi: Support vs Confidence');plt.show()
top=show.head(8).iloc[::-1]
plt.barh(top.antecedents+' → '+top.consequents,top.lift)
plt.axvline(1,linestyle='--');plt.xlabel('Lift');plt.title('8 Aturan dengan Lift Tertinggi');plt.tight_layout();plt.show()

### Tiga rekomendasi bisnis konkret
1. **Kopi Susu + Croissant:** tawarkan paket sarapan di kasir dan pada menu digital; ukur perubahan nilai transaksi.
2. **Americano + Cookies:** tampilkan cookies sebagai saran tambahan ketika pelanggan memesan Americano; uji promosi bundling.
3. **Matcha + Cheesecake:** tempatkan kedua produk berdekatan pada daftar menu dan buat paket dessert.

Ketiga rekomendasi merupakan hipotesis untuk dataset sintetis, bukan bukti permintaan pelanggan nyata. Periksa aturan serta Lift hasil eksekusi dan validasi dengan data riil sebelum penerapan.

## Kesimpulan
Ambang batas yang lebih rendah memperluas kandidat pola dan berpotensi menambah noise. Lift, leverage, dan conviction melengkapi confidence untuk mengevaluasi kekuatan hubungan. Apriori dan FP-Growth seharusnya menghasilkan itemset identik pada ambang yang sama, sedangkan kecepatan aktual harus diukur. Dataset kedai kopi memperlihatkan contoh penerapan untuk bundling, rekomendasi produk, dan penataan menu.